In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rioxarray
import xarray as xr
import os
from pathlib import Path


from rasterio.features import rasterize, geometry_window
from shapely.geometry import box, mapping

GRID_CRS = "EPSG:5070"
WHEAT_CODES = [22, 23, 24]
THRESHOLD = 11

In [3]:
"""
Local static reference-table builder for county-level wheat cells.

Mirrors the GEE Scout/Harvester grid so the per-cell CDL wheat count can be
compared against the satellite (HLS) exports, and so soil features can be
attached to each active 1 km cell.

Key design decisions (settled with Claude):

  * TWO wheat counts per cell:
      - static_wheat_pixels          : CDL wheat over the FULL 1 km box.
                                       This is the inclusion/threshold gate and
                                       the upper bound that the GEE clear-wheat
                                       `sum` must never exceed.
      - static_wheat_pixels_incounty : CDL wheat restricted to the county
                                       polygon. An HONEST feature for the model
                                       (auto-floors neighbour-dominated border
                                       cells). NEVER used as the inclusion gate,
                                       NEVER overwritten.

  * INCLUSION / soil gate = static_wheat_pixels > THRESHOLD (full box).
      Keeps clean border cells (full box can be large) but drops genuinely
      tiny cells, so soil is only computed for usable cells.

  * Satellite cell_id is built by SNAPPING the reprojected coordinate to the
    1 km grid (floor/1000*1000 + 500), NOT np.rint to the nearest metre.
    Cell centres sit 500 m from any boundary, so this is immune to the ~1 m
    round-trip error of degrees -> EPSG:5070 and matches the static cell_id
    exactly.
"""

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rasterio
from rasterio.features import rasterize, geometry_window
from shapely.geometry import box, mapping

# ----------------------------------------------------------------------------
# Globals -- CONFIRM these match your existing definitions.
# ----------------------------------------------------------------------------
GRID_CRS = "EPSG:5070"
WHEAT_CODES = [22, 23, 24]   # 22 durum, 23 spring, 24 winter wheat
THRESHOLD = 11               # strict >: keeps the GEE `static_sum > 11` rule


# ----------------------------------------------------------------------------
# 1 km grid
# ----------------------------------------------------------------------------
def build_1km_grid_for_county(county_file, county_id, id_col="id_comb"):
    county = county_file[county_file[id_col] == county_id].to_crs(GRID_CRS)

    if county.empty:
        raise ValueError(f"County {county_id} not found in column {id_col}")

    county_geom = county.geometry.union_all()

    minx, miny, maxx, maxy = county_geom.bounds

    # Align to EPSG:5070 1 km grid (same origin convention as GEE coveringGrid)
    x0 = np.floor(minx / 1000) * 1000
    y0 = np.floor(miny / 1000) * 1000
    x1 = np.ceil(maxx / 1000) * 1000
    y1 = np.ceil(maxy / 1000) * 1000

    cells = []
    for x in np.arange(x0, x1, 1000):
        for y in np.arange(y0, y1, 1000):
            geom = box(x, y, x + 1000, y + 1000)
            if geom.intersects(county_geom):
                cx = int(x + 500)
                cy = int(y + 500)
                cells.append({
                    "county_id": county_id,
                    "x_5070": cx,
                    "y_5070": cy,
                    "geometry": geom,
                })

    grid = gpd.GeoDataFrame(cells, crs=GRID_CRS)

    # lon/lat centroid for compatibility with exported CSVs
    centroids = grid.geometry.centroid
    centroid_gdf = gpd.GeoDataFrame(geometry=centroids, crs=GRID_CRS).to_crs("EPSG:4326")
    grid["longitude"] = centroid_gdf.geometry.x.values
    grid["latitude"] = centroid_gdf.geometry.y.values

    # county_geom returned so the CDL step can compute the in-county count
    return grid, county_geom


# ----------------------------------------------------------------------------
# CDL wheat counts -- both full-box and in-county in a single pass
# ----------------------------------------------------------------------------
def add_cdl_wheat_counts(grid_5070, cdl_tif, county_geom, wheat_codes=WHEAT_CODES):
    grid = grid_5070.copy()

    with rasterio.open(cdl_tif) as src:
        grid_cdl = grid.to_crs(src.crs)

        window = geometry_window(src, grid_cdl.geometry.map(mapping).tolist())
        cdl = src.read(1, window=window)
        transform = src.window_transform(window)

        # Rasterize each cell to a unique integer label (1..N)
        shapes = ((geom, idx + 1) for idx, geom in enumerate(grid_cdl.geometry))
        cell_labels = rasterize(
            shapes=shapes,
            out_shape=cdl.shape,
            transform=transform,
            fill=0,
            dtype="int32",
            all_touched=False,
        )

        # Rasterize the county polygon onto the SAME window/transform/shape.
        # all_touched=False -> a CDL pixel is "in county" iff its centre is
        # inside the polygon, consistent with the cell rasterization above.
        county_cdl = gpd.GeoSeries([county_geom], crs=GRID_CRS).to_crs(src.crs).iloc[0]
        in_county = rasterize(
            [(county_cdl, 1)],
            out_shape=cdl.shape,
            transform=transform,
            fill=0,
            dtype="uint8",
            all_touched=False,
        ).astype(bool)

        valid = cell_labels > 0
        if src.nodata is not None:
            valid &= cdl != src.nodata

        wheat_full = np.isin(cdl, wheat_codes) & valid
        wheat_in = wheat_full & in_county

        n = len(grid) + 1
        wheat_full_counts = np.bincount(cell_labels[wheat_full].ravel(), minlength=n)[1:]
        wheat_in_counts = np.bincount(cell_labels[wheat_in].ravel(), minlength=n)[1:]
        valid_counts = np.bincount(cell_labels[valid].ravel(), minlength=n)[1:]

    # Full-box count: inclusion gate + upper bound for the GEE comparison
    grid["static_wheat_pixels"] = wheat_full_counts.astype(int)
    # In-county count: honest feature, never the gate, never overwritten
    grid["static_wheat_pixels_incounty"] = wheat_in_counts.astype(int)

    grid["cdl_valid_pixels"] = valid_counts.astype(int)
    # Fraction of the cell's wheat that actually lies inside the county.
    # ~1.0 = clean interior cell; near 0 = border cell sitting in a neighbour's
    # field. Handy for inspection / optional downweighting; not a filter here.
    grid["incounty_ratio"] = (
        grid["static_wheat_pixels_incounty"]
        / grid["static_wheat_pixels"].replace(0, np.nan)
    )

    return grid


def filter_active_wheat_cells(grid, threshold=THRESHOLD):
    # Gate on the FULL-BOX count. Strict > matches GEE's `static_sum > 11`.
    return grid[grid["static_wheat_pixels"] > threshold].copy()


# ----------------------------------------------------------------------------
# SoilGrids means over each 1 km cell
# ----------------------------------------------------------------------------
def infer_xy_dims(da):
    x_candidates = ["x", "lon", "longitude"]
    y_candidates = ["y", "lat", "latitude"]
    x_dim = next((d for d in x_candidates if d in da.dims), None)
    y_dim = next((d for d in y_candidates if d in da.dims), None)
    if x_dim is None or y_dim is None:
        raise ValueError(f"Could not infer spatial dims from {da.dims}")
    return x_dim, y_dim


def prepare_soil_da(da, soil_crs=None):
    for dim in list(da.dims):
        if da.sizes[dim] == 1 and dim not in ["x", "y", "lon", "lat", "longitude", "latitude"]:
            da = da.squeeze(dim)
            
    x_dim, y_dim = infer_xy_dims(da)
    da = da.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim)

    if da.rio.crs is None:
        if soil_crs is None:
            raise ValueError("SoilGrids xarray has no CRS. Pass soil_crs='EPSG:4326' or the correct CRS.")
        da = da.rio.write_crs(soil_crs)

    return da


from rasterio.features import rasterize
from shapely.geometry import mapping
import numpy as np
import pandas as pd

def add_soil_means_fast(grid_5070, dataarray, soil_var, soil_crs="EPSG:4326", all_touched=True):
    """
    Fast version:
    - rasterizes all active 1 km cells onto the SoilGrids raster once
    - computes mean per cell with np.bincount
    - avoids per-cell rio.clip()
    """

    grid = grid_5070.copy().reset_index(drop=True)

    da = prepare_soil_da(dataarray, soil_crs=soil_crs)

    # force 2D y/x order
    x_dim = da.rio.x_dim
    y_dim = da.rio.y_dim
    da = da.transpose(y_dim, x_dim)

    values = da.values.astype("float64")

    # reproject grid to soil CRS
    grid_soil_crs = grid.to_crs(da.rio.crs).reset_index(drop=True)

    # label each 1 km cell with unique integer: 1..N
    shapes = (
        (mapping(geom), idx + 1)
        for idx, geom in enumerate(grid_soil_crs.geometry)
    )

    labels = rasterize(
        shapes=shapes,
        out_shape=values.shape,
        transform=da.rio.transform(recalc=True),
        fill=0,
        dtype="int32",
        all_touched=all_touched,
    )

    valid = labels > 0
    valid &= np.isfinite(values)

    nodata = da.rio.nodata
    if nodata is not None and not pd.isna(nodata):
        valid &= values != nodata

    n = len(grid) + 1

    sums = np.bincount(
        labels[valid].ravel(),
        weights=values[valid].ravel(),
        minlength=n,
    )

    counts = np.bincount(
        labels[valid].ravel(),
        minlength=n,
    )

    means = np.full(len(grid), np.nan, dtype="float64")

    cell_counts = counts[1:]
    cell_sums = sums[1:]

    ok = cell_counts > 0
    means[ok] = cell_sums[ok] / cell_counts[ok]

    grid[f"{soil_var}_mean_1km"] = means
    grid[f"{soil_var}_soil_pixel_count_1km"] = cell_counts.astype(int)

    return grid



def process_county_cells(
    county_file,
    county_id,
    cdl_tif,
    soil_xarray_folder,
    output_csv,
    id_col="id_comb",
    soil_vars=[None],
    threshold=THRESHOLD,
    soil_crs="EPSG:4326",
):
    grid, county_geom = build_1km_grid_for_county(
        county_file=county_file,
        county_id=county_id,
        id_col=id_col,
    )

    grid = add_cdl_wheat_counts(
        grid_5070=grid,
        cdl_tif=cdl_tif,
        county_geom=county_geom,
        wheat_codes=WHEAT_CODES,
    )

    active = filter_active_wheat_cells(grid, threshold=threshold)  
    for soil_var in soil_vars:
        Patharray =os.path.join(soil_xarray_folder, Path((county_id + "_" + soil_var + ".nc")))
        with xr.open_dataarray(Patharray) as dataarray:
            active = add_soil_means_fast(
                grid_5070=active,
                dataarray=dataarray,
                soil_var=soil_var,
                soil_crs=soil_crs,
                all_touched=True,
            )
    out = pd.DataFrame(active.drop(columns="geometry"))
    out.to_csv(output_csv, index=False)
    return out

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import traceback
import os

feature_soil = ['ocd_0-5cm_mean', 'soc_0-5cm_mean', 'nitrogen_0-5cm_mean', 'phh2o_gradient_30_100_minus_0_30', 'clay_gradient_30_100_minus_0_30', 'sand_gradient_30_100_minus_0_30', 'ocd_0_30', 'awc_0_30', 'nitrogen_gradient_30_100_minus_0_30', 'bdod_0-5cm_mean', 'soc_gradient_30_100_minus_0_30', 'soc_0_30', 'ocd_30-60cm_mean', 'phh2o_30-60cm_mean', 'wv0010_0-5cm_mean', 'nitrogen_0_30', 'nitrogen_15-30cm_mean', 'phh2o_30_100']

us_list_shapes = gpd.read_file(r'Data_Ashape/gadm36_USA_2.shp')
us_list_shapes["id_combo"] = us_list_shapes["id_comb"]
years = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2025"]

ids =['48_12', '48_24'] 
MAX_WORKERS = 50  # start with 3-4; more may just kill disk I/O

OUT_DIR = Path(r"Data_Astatic_wheat_soil_cells")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CDL_BASE = Path(r"Data_AUSDA-Cropland")
SOIL_FOLDER = r"Data_Asoil_data"


def run_one_county_year(year, county_id):
    print(f"Processing county {county_id} year {year}...")
    tif = CDL_BASE / f"{year}_30m_cdls" / f"{year}_30m_cdls.tif"
    output_csv = OUT_DIR / f"county_{county_id}_{year}_static_wheat_soil_cells.csv"

    # skip already finished files
    if output_csv.exists():
        return {
            "year": year,
            "county_id": county_id,
            "status": "skipped_exists",
            "rows": None,
            "output": str(output_csv),
        }

    result = process_county_cells(
        county_file=us_list_shapes,   # your already loaded GeoDataFrame
        county_id=county_id,
        cdl_tif=str(tif),
        soil_xarray_folder=SOIL_FOLDER,
        output_csv=str(output_csv),
        id_col="id_comb",
        soil_vars=feature_soil,
        threshold=THRESHOLD,
        soil_crs="EPSG:4326",
    )

    return {
        "year": year,
        "county_id": county_id,
        "status": "ok",
        "rows": len(result) if result is not None else None,
        "output": str(output_csv),
    }


jobs = [(year, county_id) for year in years for county_id in ids]

results = []
failed = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(run_one_county_year, year, county_id): (year, county_id)
        for year, county_id in jobs
    }

    for i, future in enumerate(as_completed(futures), start=1):
        year, county_id = futures[future]

        try:
            res = future.result()
            results.append(res)
            print(f"[{i}/{len(jobs)}] {county_id} {year}: {res['status']} rows={res['rows']}")

        except Exception as e:
            err = {
                "year": year,
                "county_id": county_id,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
            failed.append(err)
            print(f"[{i}/{len(jobs)}] FAILED {county_id} {year}: {e}")

print("Done.")
print("Successful/skipped:", len(results))
print("Failed:", len(failed))

failed[:5]

Processing county 48_12 year 2016...
Processing county 48_24 year 2016...
Processing county 48_12 year 2017...
Processing county 48_24 year 2017...
Processing county 48_12 year 2018...
Processing county 48_24 year 2018...
Processing county 48_12 year 2019...
Processing county 48_24 year 2019...
Processing county 48_12 year 2020...
Processing county 48_24 year 2020...
Processing county 48_12 year 2021...
Processing county 48_24 year 2021...
Processing county 48_12 year 2022...
Processing county 48_24 year 2022...
Processing county 48_12 year 2023...
Processing county 48_24 year 2023...
Processing county 48_12 year 2025...
Processing county 48_24 year 2025...


<home>\AppData\Roaming\Python\Python313\site-packages\rasterio\features.py:410: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _rasterize(valid_shapes, out, transform, all_touched, merge_alg)


[1/18] 48_12 2016: ok rows=1041
[2/18] 48_12 2025: ok rows=962
[3/18] 48_12 2022: ok rows=1068
[4/18] 48_12 2020: ok rows=1093
[5/18] 48_12 2018: ok rows=1063
[6/18] 48_12 2017: ok rows=1070
[7/18] 48_12 2021: ok rows=1194
[8/18] 48_12 2019: ok rows=1116
[9/18] 48_12 2023: ok rows=1126
[10/18] 48_24 2021: ok rows=318
[11/18] 48_24 2016: ok rows=297
[12/18] 48_24 2020: ok rows=304
[13/18] 48_24 2022: ok rows=352
[14/18] 48_24 2019: ok rows=278
[15/18] 48_24 2017: ok rows=224
[16/18] 48_24 2023: ok rows=360
[17/18] 48_24 2018: ok rows=244
[18/18] 48_24 2025: ok rows=349
Done.
Successful/skipped: 18
Failed: 0


[]